# 00 - Environment check

Run once per team member: Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all.
The last cell prints a pass/fail summary.

In [ ]:
# Colab's preinstalled torchao is older than what diffusers/peft expect; we don't use it.
%pip uninstall -q -y torchao
import importlib.util, sys
if "torchao" in sys.modules:
    raise RuntimeError("An old torchao is still loaded from a previous run. "
                       "Runtime > Disconnect and delete runtime, then Run all again.")
if importlib.util.find_spec("torchao") is not None:
    raise RuntimeError(f"torchao could not be removed: {importlib.util.find_spec('torchao').origin}")

%pip install -q diffusers transformers peft accelerate
import diffusers, transformers, peft
from diffusers import AutoPipelineForInpainting
print("diffusers", diffusers.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
results = {"Packages": True}

In [ ]:
import torch
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU: {torch.cuda.get_device_name(0)} | {free / 1e9:.1f} of {total / 1e9:.1f} GB free")
    results["GPU"] = True
else:
    print("No GPU. Runtime > Change runtime type > T4 GPU, then rerun.")
    results["GPU"] = False

In [ ]:
BRANCH = "main"  # set to your branch name to run a branch

import os, subprocess, yaml
REPO = "/content/genai-temp-project-name"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/AlmogSchwartzberg/genai-temp-project-name.git", REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "-q"], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", BRANCH], check=True)
subprocess.run(["git", "-C", REPO, "pull", "-q", "origin", BRANCH], check=True)
cfg = yaml.safe_load(open(f"{REPO}/config.yaml"))

from google.colab import drive
drive.mount("/content/drive")
ROOT = cfg["drive_root"]
for folder in cfg["drive_folders"]:
    os.makedirs(f"{ROOT}/{folder}", exist_ok=True)
print("Config loaded, Drive ready at", ROOT)
results["Repo, config and Drive"] = True

Inpainting repaints only the white area of the mask. This uses the standard diffusers example: a dog on a park bench is replaced by a bear.

In [ ]:
import time, json
import numpy as np
from PIL import Image
from diffusers import AutoPipelineForInpainting
from diffusers.utils import load_image

size = cfg["generation"]["image_size"]
pipe = AutoPipelineForInpainting.from_pretrained(
    cfg["generation"]["model"], torch_dtype=torch.float16, variant="fp16").to("cuda")
examples = "https://raw.githubusercontent.com/CompVis/latent-diffusion/main/data/inpainting_examples"
image = load_image(f"{examples}/overture-creations-5sI6fQgYIuo.png").resize((size, size))
mask = load_image(f"{examples}/overture-creations-5sI6fQgYIuo_mask.png").convert("L").resize((size, size))
box = list(mask.getbbox())
prompt = ("a cute brown bear cub sitting upright on a park bench, facing the camera, "
          "looking at the camera and smiling, high resolution photo")

start = time.time()
generator = torch.Generator("cuda").manual_seed(cfg["seed"])
result = pipe(prompt=prompt, image=image, mask_image=mask, generator=generator).images[0]
seconds = time.time() - start
print(f"Generated in {seconds:.1f}s")
display(Image.fromarray(np.hstack([np.array(image), np.array(mask.convert("RGB")), np.array(result)])))

x0, y0, x1, y1 = box
change = np.abs(np.array(result, dtype=float) - np.array(image, dtype=float))[y0:y1, x0:x1].mean()
print(f"Average pixel change inside the mask: {change:.0f} (0 = nothing was painted)")
assert change > 10, "The masked area stayed almost unchanged - generation did not work"
results["Generation"] = True

In [ ]:
out_dir = f"{ROOT}/synthetic/env_check"
os.makedirs(out_dir, exist_ok=True)
result.save(f"{out_dir}/sample.png")
meta = {"method": "add_objects", "prompt": prompt, "seed": cfg["seed"], "model": cfg["generation"]["model"],
        "bbox_xyxy": box, "seconds": round(seconds, 1)}
json.dump(meta, open(f"{out_dir}/sample.json", "w"), indent=2)
assert os.path.exists(f"{out_dir}/sample.png")
print("Saved to", out_dir)
results["Save to Drive"] = True

The resume pattern every long notebook uses: skip items whose output already exists in Drive, so a rerun after a disconnect continues instead of starting over.

In [ ]:
import shutil

def run_items(count, out_dir, stop_after=None):
    made = 0
    for i in range(count):
        path = f"{out_dir}/item_{i:03d}.png"
        if os.path.exists(path):
            continue
        if stop_after is not None and made == stop_after:
            raise RuntimeError("simulated disconnect")
        result.resize((64, 64)).save(path)
        made += 1
    return made

resume_dir = f"{ROOT}/checkpoints/env_check_resume"
shutil.rmtree(resume_dir, ignore_errors=True)
os.makedirs(resume_dir)
try:
    run_items(3, resume_dir, stop_after=2)
except RuntimeError as error:
    print("Interrupted:", error)
made_after_restart = run_items(3, resume_dir)
assert made_after_restart == 1, f"expected 1 new item after restart, got {made_after_restart}"
print("Resumed: only the missing item was produced")
results["Resume after disconnect"] = True

In [ ]:
from transformers import AutoImageProcessor, AutoModelForObjectDetection

del pipe
torch.cuda.empty_cache()
model_id = cfg["detection"]["models"][0]
processor = AutoImageProcessor.from_pretrained(model_id)
model = AutoModelForObjectDetection.from_pretrained(model_id).to("cuda").eval()
inputs = processor(images=result, return_tensors="pt").to("cuda")
start = time.time()
with torch.no_grad():
    outputs = model(**inputs)
detections = processor.post_process_object_detection(outputs, target_sizes=[result.size[::-1]], threshold=0.3)[0]
print(f"{model_id}: {len(detections['scores'])} detections in {time.time() - start:.2f}s "
      "(any number is fine, this only checks that the model runs)")
results["Detection model"] = True

In [ ]:
steps = ["Packages", "GPU","Repo, config and Drive", "Generation", "Save to Drive",
         "Resume after disconnect", "Detection model"]
for step in steps:
    print("✓" if results.get(step) else "✗", step)